In [20]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.io import fits

In [17]:
def distance_calculator(z, zerr, H_0type):
    c = 299792
    H0_mode = H_0type.lower()
    if H0_mode == 'local':
        d_local = c*z/73
        d_localerr = zerr*(c/73)
        return d_local, d_localerr
    if H0_mode == 'cmb':
        d_CMB = c*z/68
        d_CMBerr = zerr*(c/68)
        return d_CMB, d_CMBerr
    else:
        raise ValueError("H_0type must be either 'local' or 'cmb'")

In [68]:
d_local, d_localerr = distance_calculator(z = 0.001568, zerr = 6.67e-6, H_0type = 'local')
print(d_local, d_localerr)
d_CMB, d_CMBerr = distance_calculator(z = 0.00147, zerr = 1e-05, H_0type = 'cmb')
print(d_CMB, d_CMBerr)

6.439367890410959 0.027391953972602738
6.480797647058823 0.04408705882352942


In [26]:
with fits.open('catalog.fits') as hdul:
    # 2. Access the extension containing the table (usually index 1 or higher)
    # Hint: Use hdul.info() to find which extension has 'BinTableHDU'
    table_hdu = hdul[1] 
    
    # 3. Convert the FITS table data into a pandas DataFrame
    df = pd.DataFrame(table_hdu.data)

# 4. Display the first few rows
print(df.head())

                  objname      pgc   nsaid           specobjid  plate  \
0  2MASXJ01235631+0012308  3519080  114636  783761887031486464    696   
1  2MASXJ04575930-0612323  3689848      -1                  -1     -1   
2  2MASXJ05534289-3205449  3704923      -1                  -1     -1   
3  2MASXJ06094464-3250007  3707868      -1                  -1     -1   
4  2MASXJ07415401+2352188  3723606      -1                  -1     -1   

   fiberid    mjd  group_id          ra        dec  ...  logmass_BV  \
0      493  52209        -1   20.984826   0.208487  ...         NaN   
1       -1     -1        -1   74.497114  -6.208914  ...         NaN   
2       -1     -1        -1   88.428899 -32.095887  ...         NaN   
3       -1     -1        -1   92.436063 -32.833630  ...         NaN   
4       -1     -1        -1  115.475523  23.871764  ...         NaN   

   logmass_BR  logmass  logmass_error  logmass_src  chandra_observation  \
0         NaN    8.166          0.150          g-i         

In [34]:
df.columns

Index(['objname', 'pgc', 'nsaid', 'specobjid', 'plate', 'fiberid', 'mjd',
       'group_id', 'ra', 'dec', 'ra_nsa', 'dec_nsa', 'ra_ned', 'dec_ned',
       'd25', 'v_h', 'v_cmb', 'v_source', 'hl_obj', 'lvg_obj', 'nsa_obj',
       'sga_obj', 't_type', 'color_type', 'best_type', 'a_B_leda', 'a_g_nsa',
       'EBV_irsa', 'Bt0_leda', 'BV_color_leda', 'B_lum', 'gi_color_nsa',
       'i_lum_nsa', 'gr_color_sga', 'r_lum_sga', 'BR_color_ned', 'R_lum_ned',
       'BMag', 'gi_color', 'mag_flag', 'cf3_dist', 'cf3_dist_error',
       'zind_dist', 'zind_dist_error', 'zind_indicator', 'bestdist',
       'bestdist_error', 'bestdist_method', 'bestdist_source', 'dist_ned_flag',
       'logmass_gi', 'logmass_gr', 'logmass_BV', 'logmass_BR', 'logmass',
       'logmass_error', 'logmass_src', 'chandra_observation',
       'chandra_detection', 'log_lx', 'chandra_detection_3arcsec',
       'log_lx_3arcsec'],
      dtype='str')

In [35]:
df = df.copy()

In [40]:
from astropy.table import Table

# 1. Read the FITS table extension using Astropy's native table reader
# (Change hdu=1 if your table is in a different extension index)
astro_table = Table.read('catalog.fits', hdu=1)

# 2. Convert it directly to a pandas DataFrame 
# This automatically handles the Big-Endian to Little-Endian conversion safely
df = astro_table.to_pandas()

# 3. Now the filtering will work flawlessly without any byte-order errors!
NGC1313 = df[df['objname'] == 'NGC1313']

In [43]:
if df['objname'].dtype == object:
    # If it's a byte string, decode it first
    if isinstance(df['objname'].iloc[0], bytes):
        df['objname'] = df['objname'].str.decode('utf-8')
    
    # Strip hidden whitespace padding
    df['objname'] = df['objname'].str.strip()

# Query again
NGC1313 = df[df['objname'] == 'NGC1313']

In [45]:
NGC1313.columns

Index(['objname', 'pgc', 'nsaid', 'specobjid', 'plate', 'fiberid', 'mjd',
       'group_id', 'ra', 'dec', 'ra_nsa', 'dec_nsa', 'ra_ned', 'dec_ned',
       'd25', 'v_h', 'v_cmb', 'v_source', 'hl_obj', 'lvg_obj', 'nsa_obj',
       'sga_obj', 't_type', 'color_type', 'best_type', 'a_B_leda', 'a_g_nsa',
       'EBV_irsa', 'Bt0_leda', 'BV_color_leda', 'B_lum', 'gi_color_nsa',
       'i_lum_nsa', 'gr_color_sga', 'r_lum_sga', 'BR_color_ned', 'R_lum_ned',
       'BMag', 'gi_color', 'mag_flag', 'cf3_dist', 'cf3_dist_error',
       'zind_dist', 'zind_dist_error', 'zind_indicator', 'bestdist',
       'bestdist_error', 'bestdist_method', 'bestdist_source', 'dist_ned_flag',
       'logmass_gi', 'logmass_gr', 'logmass_BV', 'logmass_BR', 'logmass',
       'logmass_error', 'logmass_src', 'chandra_observation',
       'chandra_detection', 'log_lx', 'chandra_detection_3arcsec',
       'log_lx_3arcsec'],
      dtype='str')

In [47]:
NGC1313 = NGC1313[['objname', 'pgc', 'ra', 'dec', 'ra_nsa', 'dec_nsa', 'ra_ned', 'dec_ned',
       'd25', 'v_h', 'v_cmb', 'v_source', 't_type', 'color_type', 'best_type', 'Bt0_leda', 'BV_color_leda', 'B_lum', 'gi_color_nsa',
       'i_lum_nsa', 'gr_color_sga', 'r_lum_sga', 'BR_color_ned', 'R_lum_ned',
       'BMag', 'gi_color', 'mag_flag', 'cf3_dist', 'cf3_dist_error',
       'zind_dist', 'zind_dist_error', 'zind_indicator', 'bestdist',
       'bestdist_error', 'bestdist_method', 'bestdist_source', 'dist_ned_flag']]

In [59]:
ra = NGC1313['ra'].values
dec = NGC1313['dec'].values
print(ra, dec)

[49.5655215] [-66.4978677]


In [63]:
v_h = NGC1313['v_h'].values
v_cmb = NGC1313['v_cmb'].values
print(v_h, v_cmb)

[470.] [651.99]


In [62]:
cf3_dist = NGC1313['cf3_dist'].values
print(cf3_dist)

[3.7896109]


In [69]:
dcmb = 	6.51
dcmb_err = 0.46

In [70]:
cz_cmb = dcmb*68
cz_cmb_err = dcmb_err*68

In [71]:
cz_cmb, cz_cmb_err

(442.68, 31.28)